<a href="https://colab.research.google.com/github/Chandrani45/AI-COURSE-RECOMMENDER-PROJECT-WORK-/blob/main/Chandrani_Sengupta_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

LOADING THE LIBRARIES

In [42]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [43]:
!pip install groq

MODULE 0
Loading Groq

In [44]:
import os
from groq import Groq

In [45]:
import os
from getpass import getpass

os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")

Enter your Groq API key: ··········


In [46]:
client = Groq()

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "system", "content": "You are a concise assistant."},
        {"role": "user", "content": "Explain why Groq's inference is fast in 3 sentences."},
    ],
)

print(response.choices[0].message.content)

Groq's inference is fast due to its tensor processing unit (TPU) architecture, which is specifically designed for high-performance machine learning workloads. The TPU's unique design allows for massive parallelization of computations, enabling Groq to process large amounts of data quickly and efficiently. Additionally, Groq's compiler and software stack are optimized to take full advantage of the TPU's capabilities, further accelerating inference performance.


LOADING THE COURSE DATASET

In [47]:
course=pd.read_csv('/content/coursera_course_dataset_v3.csv')

PRINTING THE FIRST 5 ROWS

In [48]:
course.head()

,Unnamed: 0,Title,Organization,Skills,Ratings,course_url,course_students_enrolled,course_description,Review Count,Difficulty,Type,Duration
0,0,Google Cybersecurity,Google,"Network Security, Python Programming, Linux, ...",4.8,https://www.coursera.org/professional-certific...,"700,909",Google Cloud Fundamentals: Core Infrastructure...,20K,Beginner,Professional Certificate,3 - 6 Months
1,1,Google Data Analytics,Google,"Data Analysis, R Programming, SQL, Business C...",4.8,https://www.coursera.org/professional-certific...,"229,865",Prepare for a new career in the high-growth fi...,137K,Beginner,Professional Certificate,3 - 6 Months
2,3,Google Project Management:,Google,"Project Management, Strategy and Operations, ...",4.8,https://www.coursera.org/professional-certific...,"29,702",Prepare-se para uma nova carreira no campo de ...,100K,Beginner,Professional Certificate,3 - 6 Months
3,4,IBM Data Science,IBM,"Python Programming, Data Science, Machine Lea...",4.6,https://www.coursera.org/professional-certific...,"239,622",Prepare for a career in the high-growth field ...,120K,Beginner,Professional Certificate,3 - 6 Months
4,5,Google Digital Marketing & E-commerce,Google,"Digital Marketing, Marketing, Marketing Manag...",4.8,https://www.coursera.org/professional-certific...,"384,238",This course is the eighth course in the Google...,23K,Beginner,Professional Certificate,3 - 6 Months


We drop the unnamed column as it is of no use

In [49]:
course.rename(columns={'Unnamed: 0':'Course_id'},inplace=True)

In [50]:
course.head()

,Course_id,Title,Organization,Skills,Ratings,course_url,course_students_enrolled,course_description,Review Count,Difficulty,Type,Duration
0,0,Google Cybersecurity,Google,"Network Security, Python Programming, Linux, ...",4.8,https://www.coursera.org/professional-certific...,"700,909",Google Cloud Fundamentals: Core Infrastructure...,20K,Beginner,Professional Certificate,3 - 6 Months
1,1,Google Data Analytics,Google,"Data Analysis, R Programming, SQL, Business C...",4.8,https://www.coursera.org/professional-certific...,"229,865",Prepare for a new career in the high-growth fi...,137K,Beginner,Professional Certificate,3 - 6 Months
2,3,Google Project Management:,Google,"Project Management, Strategy and Operations, ...",4.8,https://www.coursera.org/professional-certific...,"29,702",Prepare-se para uma nova carreira no campo de ...,100K,Beginner,Professional Certificate,3 - 6 Months
3,4,IBM Data Science,IBM,"Python Programming, Data Science, Machine Lea...",4.6,https://www.coursera.org/professional-certific...,"239,622",Prepare for a career in the high-growth field ...,120K,Beginner,Professional Certificate,3 - 6 Months
4,5,Google Digital Marketing & E-commerce,Google,"Digital Marketing, Marketing, Marketing Manag...",4.8,https://www.coursera.org/professional-certific...,"384,238",This course is the eighth course in the Google...,23K,Beginner,Professional Certificate,3 - 6 Months


In [51]:
course.columns

Index(['Course_id', 'Title', 'Organization', 'Skills', 'Ratings', 'course_url',
       'course_students_enrolled', 'course_description', 'Review Count',
       'Difficulty', 'Type', 'Duration'],
      dtype='object')

In [52]:
course.columns=course.columns.str.lower()


In [53]:
course.head()

,course_id,title,organization,skills,ratings,course_url,course_students_enrolled,course_description,review count,difficulty,type,duration
0,0,Google Cybersecurity,Google,"Network Security, Python Programming, Linux, ...",4.8,https://www.coursera.org/professional-certific...,"700,909",Google Cloud Fundamentals: Core Infrastructure...,20K,Beginner,Professional Certificate,3 - 6 Months
1,1,Google Data Analytics,Google,"Data Analysis, R Programming, SQL, Business C...",4.8,https://www.coursera.org/professional-certific...,"229,865",Prepare for a new career in the high-growth fi...,137K,Beginner,Professional Certificate,3 - 6 Months
2,3,Google Project Management:,Google,"Project Management, Strategy and Operations, ...",4.8,https://www.coursera.org/professional-certific...,"29,702",Prepare-se para uma nova carreira no campo de ...,100K,Beginner,Professional Certificate,3 - 6 Months
3,4,IBM Data Science,IBM,"Python Programming, Data Science, Machine Lea...",4.6,https://www.coursera.org/professional-certific...,"239,622",Prepare for a career in the high-growth field ...,120K,Beginner,Professional Certificate,3 - 6 Months
4,5,Google Digital Marketing & E-commerce,Google,"Digital Marketing, Marketing, Marketing Manag...",4.8,https://www.coursera.org/professional-certific...,"384,238",This course is the eighth course in the Google...,23K,Beginner,Professional Certificate,3 - 6 Months


In [54]:
#print rows and columns
course.shape

(623, 12)

The data has 623 rows and 11 columns

In [55]:
role_skills={'Data Scientist': ['python', 'statistics', 'machine learning', 'sql',
                           'data visualization', 'deep learning'], 'Data Analyst': ['excel', 'sql', 'statistics', 'data visualization', 'python'], 'ML Engineer':['python', 'machine learning', 'deep learning', 'sql', 'cloud'],'Data Engineer': ['python','sql', 'data wrangling', 'cloud', 'big data', 'apis'], 'Business Analyst': ['excel', 'sql', 'statistics', 'data visualization',
                           'communication'],'BI Developer': ['sql', 'data visualization', 'excel', 'statistics', 'power bi'],'AI Researcher':['python', 'deep learning', 'machine learning',
                           'mathematics', 'nlp'], 'Backend Developer': ['python', 'sql', 'apis', 'cloud', 'git'],'MLOps Engineer':['python', 'machine learning', 'cloud', 'docker', 'mlops'], 'NLP Engineer':['python', 'machine learning', 'deep learning', 'nlp',
                           'statistics']



}

In [56]:
role_skills['Data Scientist']

['python',
 'statistics',
 'machine learning',
 'sql',
 'data visualization',
 'deep learning']

In [57]:
course.isnull().sum()

,0
course_id,0
title,0
organization,0
skills,0
ratings,0
course_url,220
course_students_enrolled,236
course_description,221
review count,0
difficulty,0


In [58]:
course.duplicated().sum()

np.int64(0)

No duplicates as such

In [59]:
course.drop(columns=['organization','ratings','course_url' ,	'course_students_enrolled' ,'course_description', 'review count','difficulty','type','duration'],axis=1,inplace=True)

In [60]:
course.isnull().sum()

,0
course_id,0
title,0
skills,0


In [61]:
course.head()

,course_id,title,skills
0,0,Google Cybersecurity,"Network Security, Python Programming, Linux, ..."
1,1,Google Data Analytics,"Data Analysis, R Programming, SQL, Business C..."
2,3,Google Project Management:,"Project Management, Strategy and Operations, ..."
3,4,IBM Data Science,"Python Programming, Data Science, Machine Lea..."
4,5,Google Digital Marketing & E-commerce,"Digital Marketing, Marketing, Marketing Manag..."


In [62]:
course.shape

(623, 3)

In [76]:
#cleaning the skills dataset
import re
def clean(text):
  return re.sub(r'[^a-z0-9]','',text.str.lower())
  return re.sub(r'\s+',"",text).strip()

In [79]:
import re

def clean(s):
    s = re.sub(r"[^a-z0-9 ]", " ", str(s).lower())
    return re.sub(r"\s+", " ", s).strip()

# build the search text from Title + Skills  (note the " " between them)
course['text'] = (course['title'] + " " + course['skills']).apply(clean)

# keep a clean LIST of skills — M6 needs this
course['skills'] = course['skills'].fillna("").apply(
    lambda s: [x.strip().lower() for x in str(s).split(",") if x.strip()])

course[['course_id', 'title', 'skills', 'text']].head(3)

,course_id,title,skills,text
0,0,Google Cybersecurity,"[network security, python programming, linux, ...",google cybersecurity network security python p...
1,1,Google Data Analytics,"[data analysis, r programming, sql, business c...",google data analytics data analysis r programm...
2,3,Google Project Management:,"[project management, strategy and operations, ...",google project management project management s...


In [ ]:
course.head()

In [ ]:
course['text'].iloc[0]

In [81]:
target_role = input('Enter a role: ').strip()
assert target_role in role_skills, f"'{target_role}' is not in the role list. Pick one of: {list(role_skills)}"

Enter a role: Data Scientist


In [82]:
target_role

'Data Scientist'

In [83]:
current_skills = [s.strip().lower() for s in input('Enter your current skills (comma-separated): ').split(",") if s.strip()]
have = set(current_skills)


Enter your current skills (comma-separated): python,statistics


In [85]:
current_skills

['python', 'statistics']

In [86]:
skill_gap = [s for s in role_skills[target_role] if s.lower() not in have]
query_text = " ".join(skill_gap)
print("skill_gap:", skill_gap)

skill_gap: ['machine learning', 'sql', 'data visualization', 'deep learning']


In [87]:
query_text

'machine learning sql data visualization deep learning'

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [89]:
tfidf=TfidfVectorizer(stop_words='english')

In [96]:
vectors=tfidf.fit_transform(skill_gap).toarray()

In [97]:
from sklearn.metrics.pairwise import cosine_similarity

In [98]:
cosine_similarity(vectors)

array([[1.        , 0.        , 0.        , 0.38332232],
       [0.        , 1.        , 0.        , 0.        ],
       [0.        , 0.        , 1.        , 0.        ],
       [0.38332232, 0.        , 0.        , 1.        ]])